# 🎥 Object Tracking: YOLO + ByteTrack / BoT-SORT

**Практическая работа в Google Colab** · вводный уровень

В этом ноутбуке вы сможете:
1. Загрузить **собственное видео** (MP4, AVI, MOV и др.) или воспользоваться демонстрационным роликом.
2. Сравнить несколько **детекторов**: YOLO26n, YOLO26s и YOLO11n (опционально YOLO26n-seg).
3. Переключать **алгоритмы трекинга**: ByteTrack и BoT-SORT.
4. Визуализировать bounding boxes, имена классов, **Track ID** и траектории.
5. Скачать обработанные видео и сравнить скорость и количество выданных ID.

> **Важно:** количество уникальных ID — **не метрика качества трекинга**. Один объект может получить несколько ID при ошибках ассоциации. Для объективного сравнения нужны размеченные траектории и метрики IDF1/HOTA и др.

[Официальная документация Ultralytics: Tracking](https://docs.ultralytics.com/modes/track)

## 1. Установка и проверка GPU
Запустите ячейку. В Colab желательно выбрать **Runtime → Change runtime type → T4 GPU**. На CPU тоже работает, но медленнее.

In [ ]:
%pip -q install -U ultralytics opencv-python-headless pandas tqdm

In [ ]:
import sys, torch, ultralytics
print("Python:", sys.version.split()[0])
print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Загрузите своё видео
По умолчанию откроется окно загрузки файла. Если хотите сначала попробовать демонстрационный ролик, установите `USE_DEMO = True`.

**Рекомендация:** для первого запуска — видео 10–30 секунд, 720p, с движущимися людьми или машинами. Если видео очень длинное, ниже можно ограничить обработку первыми 150 кадрами.

In [ ]:
from pathlib import Path
from google.colab import files
import urllib.request

INPUT_DIR = Path('/content/input')
OUTPUT_DIR = Path('/content/tracking_results')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_DEMO = False  # True: скачать демонстрационное видео; False: загрузить своё

if USE_DEMO:
    # Открытый пример Ultralytics; если ссылка недоступна, используйте загрузку своего видео.
    video_path = INPUT_DIR / 'demo.mp4'
    urllib.request.urlretrieve('https://github.com/ultralytics/assets/releases/download/v0.0.0/solutions_ci_demo.mp4', video_path)
else:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('Не загружено ни одного файла.')
    filename, content = next(iter(uploaded.items()))
    video_path = INPUT_DIR / Path(filename).name
    video_path.write_bytes(content)

print('Video:', video_path)
print('Size:', f'{video_path.stat().st_size/1024/1024:.1f} MB')

### Посмотрим исходное видео

In [ ]:
import cv2
from IPython.display import Video, display

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f'OpenCV не может открыть видео: {video_path}. Попробуйте MP4 (H.264).')
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f'Resolution: {width}×{height} | FPS: {fps:.1f} | Frames: {frame_count}')
# Для некоторых кодеков браузерное воспроизведение исходника может быть недоступно.
display(Video(str(video_path), embed=True, width=640))

## 3. Настройки эксперимента
**Детектор** определяет, где объекты на каждом кадре. **Трекер** присваивает им постоянные ID, используя информацию между кадрами.

Мы сравниваем детекторы при одинаковых параметрах обработки. Для воспроизводимости каждый запуск создаёт **новый экземпляр модели**: состояния трекера не смешиваются между экспериментами.

Можно менять список `EXPERIMENTS`:

In [ ]:
# Названия весов моделей Ultralytics
# YOLO26n — быстрый baseline; YOLO26s — более крупный детектор;
# YOLO11n — другая модель для сравнения поколений;
# YOLO26n-seg — instance segmentation + tracking (опционально).
EXPERIMENTS = [
    ('yolo26n.pt', 'bytetrack.yaml'),
    ('yolo26s.pt', 'bytetrack.yaml'),
    ('yolo11n.pt', 'bytetrack.yaml'),
    ('yolo26n.pt', 'botsort.yaml'),
    # ('yolo26n-seg.pt', 'bytetrack.yaml'),
]

MAX_FRAMES = 150       # None -> всё видео; 150 для быстрой демонстрации
IMGSZ = 640
CONF = 0.15            # Порог детекции; низкие пороги полезны ByteTrack
TRACK_HISTORY = 30     # Длина траектории в кадрах
SHOW_TRAILS = True
MAX_VIDEO_WIDTH = 960  # Уменьшаем кадр перед обработкой для скорости

# None -> все классы; [0] -> только люди, [2, 3, 5, 7] -> транспорт COCO
CLASSES = None


## 4. Функции для видео, ID и траекторий
В цикле вызываем `model.track(frame, persist=True)`: это позволяет сохранять состояние трекера между последовательными кадрами.

Все координаты и линии рисуем **на том же кадре**, что передали в модель. Для каждого объекта используем стабильный цвет, вычисляемый из Track ID.

In [ ]:
from collections import defaultdict, deque
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO


def track_color(track_id: int):
    # BGR: постоянный цвет для данного Track ID
    k = int(track_id)
    return ((53 * k + 101) % 200 + 40,
            (97 * k + 41) % 200 + 40,
            (131 * k + 17) % 200 + 40)


def open_video_writer(path: Path, fps: float, shape):
    h, w = shape[:2]
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    if not writer.isOpened():
        raise RuntimeError(f'Не удалось создать видео: {path}')
    return writer


def run_tracking(video_path, detector='yolo26n.pt', tracker='bytetrack.yaml',
                 max_frames=150, conf=0.15, imgsz=640, max_width=960,
                 trail_length=30, classes=None, show_trails=True):
    """Возвращает путь к MP4 и статистику одного эксперимента."""
    # ВАЖНО: новый экземпляр YOLO для каждого эксперимента!
    model = YOLO(detector)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Невозможно открыть: {video_path}')

    source_fps = cap.get(cv2.CAP_PROP_FPS)
    source_fps = source_fps if np.isfinite(source_fps) and source_fps > 0 else 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    limit = min(total, max_frames) if max_frames is not None and total > 0 else max_frames
    history = defaultdict(lambda: deque(maxlen=trail_length))
    seen_ids = set()
    num_detections = 0
    frames_done = 0
    total_duration = 0.0
    writer = None

    tag = f'{Path(detector).stem}__{Path(tracker).stem}'
    raw_path = OUTPUT_DIR / f'{tag}_raw.mp4'
    ready_path = OUTPUT_DIR / f'{tag}.mp4'

    try:
        with tqdm(total=limit, desc=tag) as bar:
            while True:
                if max_frames is not None and frames_done >= max_frames:
                    break
                ok, frame = cap.read()
                if not ok:
                    break

                h, w = frame.shape[:2]
                if max_width and w > max_width:
                    new_h = max(2, int(round(h * max_width / w)))
                    frame = cv2.resize(frame, (max_width, new_h))

                if writer is None:
                    writer = open_video_writer(raw_path, source_fps, frame.shape)

                t0 = time.perf_counter()
                result = model.track(
                    frame,
                    persist=True,
                    tracker=tracker,
                    conf=conf,
                    imgsz=imgsz,
                    classes=classes,
                    verbose=False,
                )[0]
                total_duration += time.perf_counter() - t0

                # Ultralytics рисует bbox / маски / class / ID (если ID есть)
                annotated = result.plot()

                boxes = result.boxes
                if boxes is not None:
                    num_detections += len(boxes)
                    if boxes.id is not None:
                        ids = boxes.id.int().cpu().tolist()
                        centers = boxes.xywh.cpu().numpy()[:, :2]
                        for track_id, center in zip(ids, centers):
                            seen_ids.add(track_id)
                            point = tuple(np.asarray(center, dtype=int))
                            history[track_id].append(point)
                            if show_trails and len(history[track_id]) >= 2:
                                cv2.polylines(
                                    annotated,
                                    [np.asarray(history[track_id], dtype=np.int32)],
                                    isClosed=False,
                                    color=track_color(track_id),
                                    thickness=2,
                                )
                cv2.putText(annotated, f'{Path(detector).stem} | {Path(tracker).stem}',
                            (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                writer.write(annotated)
                frames_done += 1
                bar.update(1)
    finally:
        cap.release()
        if writer is not None:
            writer.release()

    if frames_done == 0:
        raise RuntimeError('Нет кадров для обработки.')

    # mp4v часто не проигрывается в браузере: перекодируем в H.264 (ffmpeg в Colab).
    import subprocess
    command = [
        'ffmpeg', '-y', '-loglevel', 'error', '-i', str(raw_path),
        '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(ready_path),
    ]
    subprocess.run(command, check=True)
    raw_path.unlink(missing_ok=True)

    summary = {
        'detector': detector,
        'tracker': tracker,
        'frames': frames_done,
        'unique_track_ids': len(seen_ids),
        'detections_per_frame': round(num_detections / frames_done, 2),
        'model_track_fps': round(frames_done / total_duration, 2) if total_duration else 0.0,
        'video_path': str(ready_path),
    }
    return ready_path, summary

## 5. Запуск нескольких детекторов и трекеров
Первый запуск каждого веса требует скачивания модели. Последующие обычно быстрее. Время обработки зависит от GPU, разрешения и длительности видео.

In [ ]:
summaries = []
output_videos = []

for detector, tracker in EXPERIMENTS:
    output_path, stats = run_tracking(
        video_path=video_path,
        detector=detector,
        tracker=tracker,
        max_frames=MAX_FRAMES,
        conf=CONF,
        imgsz=IMGSZ,
        max_width=MAX_VIDEO_WIDTH,
        trail_length=TRACK_HISTORY,
        classes=CLASSES,
        show_trails=SHOW_TRAILS,
    )
    output_videos.append(output_path)
    summaries.append(stats)

summary_df = pd.DataFrame(summaries)
display(summary_df.drop(columns=['video_path']))

## 6. Просмотр результатов
Выберите эксперимент и посмотрите обработанное видео с Track ID. Это видео можно скачать в следующем разделе.

In [ ]:
from IPython.display import Video, display

# Поменяйте индекс: 0, 1, 2, ...
EXPERIMENT_TO_SHOW = 0
assert 0 <= EXPERIMENT_TO_SHOW < len(output_videos)
display(Video(str(output_videos[EXPERIMENT_TO_SHOW]), embed=True, width=800))

## 7. Сравнение на одном кадре
Три изображения рядом: оригинал и результаты **двух разных экспериментов**. Сравниваем не номера ID (они назначаются независимо), а найденные объекты и качество ассоциации.

In [ ]:
import matplotlib.pyplot as plt


def read_frame_at(path, frame_number=40):
    capture = cv2.VideoCapture(str(path))
    capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_number))
    ok, bgr = capture.read()
    capture.release()
    if not ok:
        raise ValueError(f'Невозможно прочитать кадр {frame_number} из {path}')
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

FRAME_TO_COMPARE = 40
if len(output_videos) >= 2:
    frame_index = min(FRAME_TO_COMPARE, min(s['frames'] for s in summaries) - 1)
    panels = [
        ('Original', read_frame_at(video_path, frame_index)),
        (f'{summaries[0]["detector"]} + {summaries[0]["tracker"]}', read_frame_at(output_videos[0], frame_index)),
        (f'{summaries[1]["detector"]} + {summaries[1]["tracker"]}', read_frame_at(output_videos[1], frame_index)),
    ]
    fig, axes = plt.subplots(1, len(panels), figsize=(19, 6))
    for ax, (title, img) in zip(axes, panels):
        ax.imshow(img)
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Для сравнения выберите хотя бы два эксперимента в EXPERIMENTS.')

## 8. Скачать видео и таблицу сравнения

In [ ]:
from google.colab import files

summary_csv = OUTPUT_DIR / 'experiment_summary.csv'
summary_df.to_csv(summary_csv, index=False)

# Скачайте отдельное видео:
files.download(str(output_videos[0]))

# А для скачивания всех результатов одним архивом раскомментируйте:
# import shutil
# archive = shutil.make_archive('/content/tracking_results', 'zip', OUTPUT_DIR)
# files.download(archive)

## 9. Задания для студентов

**Задание A — разные детекторы.** Сравните YOLO26n, YOLO26s и YOLO11n с **одним** ByteTrack: количество детекций в среднем на кадр, время обработки, визуально заметные пропуски и неверные детекции. Объясните, почему более крупная модель не обязательно всегда лучше в конкретной сцене.

**Задание B — разные трекеры.** Сравните **ByteTrack** и **BoT-SORT** при **одном** YOLO26n. Найдите фрагмент с пересечением или перекрытием объектов. Где ID сохраняется лучше? Замечаете ли ID switches?

**Задание C — влияние порога.** Поэкспериментируйте с `CONF = 0.10`, `0.25`, `0.50`. Как меняется число найденных объектов и устойчивость треков? Учтите, что высокие пороги могут скрыть слабые детекции, полезные для ByteTrack.

**Задание D (дополнительно) — маски.** Раскомментируйте `('yolo26n-seg.pt', 'bytetrack.yaml')` и сравните tracking с bbox и instance segmentation. Чем отличается визуализация, а чем — сам процесс ассоциации?

**Вопросы для обсуждения:**
- Почему один человек может получить несколько Track ID?
- Почему объект иногда виден на кадре, но не получает подтверждённый трек?
- В каких условиях имеет смысл использовать ReID?
- Что обязательно нужно разметить, чтобы измерить IDF1/HOTA, а не просто сравнивать число ID?

### Что здесь **не** измеряется
- `unique_track_ids` не равно числу реальных уникальных объектов.
- `model_track_fps` включает вызов модели и трекера, но не перекодирование MP4 и не весь код визуализации/записи; это учебная оценка скорости, а не полноценный benchmark.
- Настоящую точность MOT оценивают по размеченным видео и согласованному протоколу.

### Справочные материалы
- [Ultralytics Tracking Mode](https://docs.ultralytics.com/modes/track)
- [Ultralytics Instance Segmentation and Tracking](https://docs.ultralytics.com/guides/instance-segmentation-and-tracking)
- [ByteTrack paper](https://arxiv.org/abs/2110.06864)
- [BoT-SORT paper](https://arxiv.org/abs/2206.14651)
